Project Overview
This Colab notebook builds a complete, lightweight prototype for detecting drunk/unfit driving using audio (speech analysis) and motion (gait stability) signals. If unfit, it simulates ignition lock and triggers SOS alerts (logs + mock SMS).

Key Features:

Synthetic data generation (no real datasets needed).
Fast ML models: Audio CNN (MFCCs for slurred/stressed/"help" detection), Motion LSTM (time-series for unstable gait).
Fusion: Weighted scores → Decision (threshold 0.7 for unfit).
SOS: Logs to file, prints simulated alerts.
Demo UI: Gradio interface (upload audio, select motion, see results + alerts).
Hackathon Impact: Multi-modal detection reduces false positives; local processing ensures privacy. Extensible to real car sensors (CAN bus, phone mic/accel).

Run all cells top-to-bottom. Training: <5 min on CPU. Demo: Interactive Gradio app with share link.

In [1]:
!pip install --upgrade gradio --quiet
!pip install numpy soundfile pillow --quiet

In [2]:
# Cell 1: Install Dependencies
# Installs all required packages (lightweight, CPU-friendly)
!pip install gradio librosa numpy pandas scikit-learn tensorflow matplotlib -q

# Imports for the notebook
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
from scipy.io import wavfile
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Flatten
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from datetime import datetime
import json
import gradio as gr
import warnings
warnings.filterwarnings('ignore')

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Create output dirs
os.makedirs('/content/assets', exist_ok=True)
os.makedirs('/content/models', exist_ok=True)

print("Dependencies installed! Ready for data generation.")


Dependencies installed! Ready for data generation.


In [3]:
# Cell 3: Synthetic Data Generation
# Generate mock data: 40 audio clips (10 each class) + 40 motion sequences (20 safe/unstable)

print("🔄 Generating synthetic data...")

# Audio: 1s clips @ 16kHz
sr = 16000
duration = 1.0
audio_files = {}
audio_features = []  # MFCCs for training
labels_audio = []    # 0:normal, 1:slurred, 2:stressed, 3:help

classes = ['normal', 'slurred', 'stressed', 'help']
for class_idx, cls in enumerate(classes):
    for i in range(10):
        t = np.linspace(0, duration, int(sr * duration))

        if cls == 'normal':
            audio = np.sin(2 * np.pi * 100 * t) * np.random.normal(1, 0.1, len(t))  # Steady tone
        elif cls == 'slurred':
            audio = np.sin(2 * np.pi * (100 + 20 * np.sin(2 * np.pi * 5 * t)) * t) * np.random.normal(1, 0.5, len(t))  # Modulated (slurry)
        elif cls == 'stressed':
            audio = np.sin(2 * np.pi * 200 * t) * np.random.normal(1, 0.3, len(t)) * (1 + 0.5 * np.sin(2 * np.pi * 10 * t))  # High freq/variance
        else:  # help
            audio = np.sin(2 * np.pi * 500 * t) * np.hanning(len(t))  # High pitch fade

        # Save sample files (first 1 per class for demo)
        if i == 0:
            filename = f'/content/assets/{cls}.wav'
            wavfile.write(filename, sr, audio.astype(np.float32))
            audio_files[cls] = filename

        # Extract MFCC features for training
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13, hop_length=512)
        mfcc_mean = np.mean(mfcc.T, axis=0)  # Average over time
        audio_features.append(mfcc_mean)
        labels_audio.append(class_idx)

# Motion: 100 timesteps x 3-axis (accel x,y,z)
time_steps = np.linspace(0, 10, 100)
motion_sequences = []
labels_motion = []

# 20 Safe (low variance)
for _ in range(20):
    safe = np.column_stack([
        np.sin(time_steps) + np.random.normal(0, 0.1, 100),
        np.cos(time_steps) + np.random.normal(0, 0.1, 100),
        np.random.normal(0, 0.05, 100)
    ])
    motion_sequences.append(safe)
    labels_motion.append(0)  # Safe

# 20 Unfit (high variance/jerky)
for _ in range(20):
    unfit = np.column_stack([
        np.sin(time_steps * 1.5) + np.random.normal(0, 0.5, 100),
        np.cos(time_steps * 0.5) + np.random.normal(0, 0.5, 100),
        np.random.normal(0, 0.3, 100)
    ])
    motion_sequences.append(unfit)
    labels_motion.append(1)  # Unfit

# Convert to arrays
X_audio = np.array(audio_features).reshape(-1, 13, 1)  # For CNN: (samples, features, 1)
y_audio = tf.keras.utils.to_categorical(labels_audio, 4)
X_motion = np.array(motion_sequences)  # (samples, timesteps, 3)
y_motion = np.array(labels_motion)

# Split for training
X_audio_train, X_audio_test, y_audio_train, y_audio_test = train_test_split(X_audio, y_audio, test_size=0.2, random_state=42)
X_motion_train, X_motion_test, y_motion_train, y_motion_test = train_test_split(X_motion, y_motion, test_size=0.2, random_state=42)

print(f"Data generated: {len(X_audio)} audio samples, {len(X_motion)} motion sequences")
print(f"Sample files: {list(audio_files.keys())}")
!ls /content/assets/

🔄 Generating synthetic data...
Data generated: 40 audio samples, 40 motion sequences
Sample files: ['normal', 'slurred', 'stressed', 'help']
help.wav  normal.wav  slurred.wav  stressed.wav


In [4]:
# Cell 5: Train Audio CNN
print("🔄 Training Audio CNN...")

# Simple CNN for MFCCs
audio_model = Sequential([
    Conv1D(32, 3, activation='relu', input_shape=(13, 1)),
    MaxPooling1D(2),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(4, activation='softmax')  # 4 classes
])

audio_model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
history_audio = audio_model.fit(X_audio_train, y_audio_train, epochs=10, batch_size=8,
                                validation_data=(X_audio_test, y_audio_test), verbose=1)

# Evaluate
y_pred_audio = audio_model.predict(X_audio_test)
acc_audio = accuracy_score(np.argmax(y_audio_test, axis=1), np.argmax(y_pred_audio, axis=1))
print(f"Audio CNN Accuracy: {acc_audio:.2f}")

# Save model
audio_model.save('/content/models/audio_cnn.h5')
print("Audio model saved!")


🔄 Training Audio CNN...
Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 155ms/step - accuracy: 0.3000 - loss: 4.5252 - val_accuracy: 0.7500 - val_loss: 0.5059
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.7500 - loss: 0.4886 - val_accuracy: 0.7500 - val_loss: 0.2638
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.9417 - loss: 0.2009 - val_accuracy: 0.8750 - val_loss: 0.3463
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step - accuracy: 0.9417 - loss: 0.1428 - val_accuracy: 1.0000 - val_loss: 0.1869
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 1.0000 - loss: 0.0749 - val_accuracy: 1.0000 - val_loss: 0.0705
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 1.0000 - loss: 0.0621 - val_accuracy: 1.0000 - val_loss: 0.0522
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 1.0000 - loss: 0.0465 - val_accuracy: 1.0000 - val_loss: 0.0305
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 63ms/step - accuracy: 1.0000 - loss: 0.0235 - val_accuracy: 1

Audio CNN Accuracy: 1.00
Audio model saved!


In [5]:
# Cell 6: Train Motion LSTM
print("🔄 Training Motion LSTM...")

# Simple LSTM for time-series
motion_model = Sequential([
    LSTM(50, input_shape=(100, 3), return_sequences=False),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')  # Binary: safe/unfit
])

motion_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
history_motion = motion_model.fit(X_motion_train, y_motion_train, epochs=10, batch_size=8,
                                  validation_data=(X_motion_test, y_motion_test), verbose=1)

# Evaluate
y_pred_motion = (motion_model.predict(X_motion_test) > 0.5).astype(int)
acc_motion = accuracy_score(y_motion_test, y_pred_motion)
print(f"Motion LSTM Accuracy: {acc_motion:.2f}")

# Save model
motion_model.save('/content/models/motion_lstm.h5')
print("Motion model saved!")

🔄 Training Motion LSTM...
Epoch 1/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 819ms/step - accuracy: 0.6583 - loss: 0.6701 - val_accuracy: 1.0000 - val_loss: 0.6308
Epoch 2/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 257ms/step - accuracy: 1.0000 - loss: 0.6018 - val_accuracy: 1.0000 - val_loss: 0.5769
Epoch 3/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 245ms/step - accuracy: 1.0000 - loss: 0.5426 - val_accuracy: 1.0000 - val_loss: 0.5182
Epoch 4/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 135ms/step - accuracy: 1.0000 - loss: 0.4798 - val_accuracy: 1.0000 - val_loss: 0.4562
Epoch 5/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 140ms/step - accuracy: 1.0000 - loss: 0.4134 - val_accuracy: 1.0000 - val_loss: 0.3862
Epoch 6/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 157ms/step - accuracy: 1.0000 - loss: 0.3387 - val_accuracy: 1.0000 - val_loss: 0.2995
Epoch 7/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 148ms/step - accuracy: 1.0000 - loss: 0.2543 - val_accuracy: 1.0000 - val_loss: 0.2066
Epoch 8/10
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 1.0000 - loss: 0.1714 - val_acc

Motion LSTM Accuracy: 1.00
Motion model saved!


In [6]:
# Cell 8: Fusion Engine
# Load models
audio_model = tf.keras.models.load_model('/content/models/audio_cnn.h5')
motion_model = tf.keras.models.load_model('/content/models/motion_lstm.h5')

def extract_audio_features(audio_path):
    """Extract MFCC from WAV"""
    y, sr = librosa.load(audio_path, sr=16000, duration=1.0)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13, hop_length=512)
    return np.mean(mfcc.T, axis=0).reshape(1, 13, 1)

def predict_audio(audio_path):
    """Predict audio classes and unfit prob"""
    features = extract_audio_features(audio_path)
    probs = audio_model.predict(features, verbose=0)[0]
    class_names = ['normal', 'slurred', 'stressed', 'help']
    unfit_prob = 1 - probs[0]  # Unfit = non-normal
    help_detected = probs[3] > 0.5
    return unfit_prob, probs, help_detected, class_names

def predict_motion(motion_seq):
    """Predict motion unfit prob (motion_seq: (100,3) array)"""
    seq = motion_seq.reshape(1, 100, 3)
    unfit_prob = motion_model.predict(seq, verbose=0)[0][0]
    return unfit_prob

def fusion_decide(audio_path, motion_type='stable'):
    """Main fusion: audio + motion → decision"""
    # Audio prediction
    audio_unfit, audio_probs, help_detected, class_names = predict_audio(audio_path)

    # Mock motion sequence
    if motion_type == 'stable':
        time_steps = np.linspace(0, 10, 100)
        motion_seq = np.column_stack([
            np.sin(time_steps) + np.random.normal(0, 0.1, 100),
            np.cos(time_steps) + np.random.normal(0, 0.1, 100),
            np.random.normal(0, 0.05, 100)
        ])
    else:  # unstable
        time_steps = np.linspace(0, 10, 100)
        motion_seq = np.column_stack([
            np.sin(time_steps * 1.5) + np.random.normal(0, 0.5, 100),
            np.cos(time_steps * 0.5) + np.random.normal(0, 0.5, 100),
            np.random.normal(0, 0.3, 100)
        ])

    motion_unfit = predict_motion(motion_seq)

    # Fusion
    drunk_score = 0.6 * audio_unfit + 0.4 * motion_unfit
    is_unfit = (drunk_score >= 0.7) or help_detected
    decision = "IGNITION LOCKED (SIMULATION)" if is_unfit else "Safe to Drive"

    return {
        'audio_unfit': audio_unfit,
        'motion_unfit': motion_unfit,
        'drunk_score': drunk_score,
        'help_detected': help_detected,
        'is_unfit': is_unfit,
        'decision': decision,
        'audio_probs': dict(zip(class_names, audio_probs))
    }

print("Fusion engine ready!")

Fusion engine ready!


In [7]:
# Cell 10: SOS Alert System (Updated: Dynamic GPS Support – Run This to Fix and Enable GPS)
import json
from datetime import datetime
import shutil
import requests  # For Fast2SMS API calls

def trigger_sos(result, audio_path, gps_location="19.0760, 72.8777", enable_sms=False, fast2sms_api_key=None, from_phone=None, to_phone=None):
    """Handles alerts: Logs data with dynamic GPS, shows mock SMS, and sends real SMS if set up."""
    # Create a timestamp for the log
    timestamp = datetime.now().isoformat()

    # Prepare clean log data (with dynamic GPS)
    log_entry = {
        'timestamp': timestamp,
        'drunk_score': float(result['drunk_score']),
        'is_unfit': bool(result['is_unfit']),
        'help_detected': bool(result['help_detected']),
        'decision': str(result['decision']),
        'gps': gps_location  # Dynamic: User-provided (e.g., "28.6139, 77.2090" for Delhi)
    }

    # Save log to a file
    with open('/content/sos_log.txt', 'a') as f:
        f.write(json.dumps(log_entry) + '\n')

    # Alert message with dynamic GPS
    alert_msg = f"ALERT: Possible impaired driving detected at GPS: {gps_location}"
    if result['help_detected']:
        alert_msg += " | HIGH PRIORITY: 'Help' detected – Contact emergency services immediately."
    print(alert_msg)

    # SMS logic (includes dynamic GPS)
    if enable_sms and fast2sms_api_key and to_phone:
        try:
            url = "https://www.fast2sms.com/dev/bulkV2"
            params = {
                'authorization': fast2sms_api_key,
                'route': 'qt',
                'message': alert_msg + f" | Impairment Score: {result['drunk_score']:.1f}",
                'numbers': to_phone
            }
            response = requests.get(url, params=params)
            if response.status_code == 200 and response.json().get('return', False):
                print(f"SMS sent successfully to {to_phone} via Fast2SMS (includes GPS: {gps_location}).")
            else:
                print("SMS delivery failed – Check API key and number.")
        except Exception as e:
            print(f"SMS error: {str(e)} – Using mock alert only.")
    else:
        print("Mock SMS: Alert would be sent to emergency contact with GPS details.")

    # Save evidence if unfit
    if result['is_unfit']:
        safe_name = f'/content/evidence_{timestamp.replace(":", "_")}.wav'
        shutil.copy(audio_path, safe_name)
        print(f"Evidence preserved: {safe_name} (Download from Colab Files).")

# Quick Test (with dynamic GPS)
test_result = {'drunk_score': 0.91, 'is_unfit': True, 'help_detected': True, 'decision': 'IGNITION LOCKED (SIMULATION)'}
trigger_sos(test_result, '/content/assets/help.wav', gps_location="28.6139, 77.2090")  # Example: Delhi
print("\nUpdated log (run !cat /content/sos_log.txt):")
!cat /content/sos_log.txt
print("✅ Dynamic GPS enabled! Now compatible with Cell 12.")


ALERT: Possible impaired driving detected at GPS: 28.6139, 77.2090 | HIGH PRIORITY: 'Help' detected – Contact emergency services immediately.
Mock SMS: Alert would be sent to emergency contact with GPS details.
Evidence preserved: /content/evidence_2025-09-26T23_26_17.582303.wav (Download from Colab Files).

Updated log (run !cat /content/sos_log.txt):
{"timestamp": "2025-09-26T19:04:39.244062", "drunk_score": 0.8, "is_unfit": true, "help_detected": true, "decision": "IGNITION LOCKED (SIMULATION)", "gps": "37.7749, -122.4194"}
{"timestamp": "2025-09-26T19:52:25.863734", "drunk_score": 0.8, "is_unfit": true, "help_detected": true, "decision": "IGNITION LOCKED (SIMULATION)", "gps": "37.7749, -122.4194"}
{"timestamp": "2025-09-26T19:56:58.838883", "drunk_score": 0.800000011920929, "is_unfit": true, "help_detected": true, "decision": "IGNITION LOCKED (SIMULATION)", "gps": "37.7749, -122.4194"}
{"timestamp": "2025-09-26T19:57:03.318556", "drunk_score": 0.800000011920929, "is_unfit": true, "

In [18]:
# Cell 12: Car-Integrated Drunk Driving Detection (Voice + Camera + Sensors – Full User-Centric)
# Run !pip install numpy soundfile --quiet first if needed.

import gradio as gr
from datetime import datetime
import random  # For simulating sensors/camera analysis
import base64  # For handling image data (mock)
import io  # For PIL to bytes conversion
import numpy as np  # For beep sound generation
import soundfile as sf  # For saving audio to WAV

# Mock trigger_sos (if not defined elsewhere – logs + optional SMS)
def trigger_sos(result, audio_path, gps_location, enable_sms, fast2sms_api_key, from_phone, to_phone):
    print(f"🚨 SOS TRIGGERED: Risk {result['drunk_score']:.0%} at {gps_location}. Help detected: {result['help_detected']}")
    if enable_sms and fast2sms_api_key and to_phone:
        try:
            import requests
            url = "https://www.fast2sms.com/dev/bulkV2"
            payload = {
                "authorization": fast2sms_api_key,
                "message": f"🚨 Car Alert: {result['drunk_score']:.0%} risk (drunk/drowsy) at GPS {gps_location}. Call now!",
                "numbers": to_phone,
                "route": "q"
            }
            response = requests.post(url, data=payload)
            print(f"SMS sent: {response.status_code}")
        except Exception as e:
            print(f"SMS error: {e}")
    # Log to file (for Colab)
    with open("/content/sos_log.txt", "a") as f:
        f.write(f"{datetime.now()}: {result}\n")

# Generate beep sound for alerts (short warning tone)
def generate_beep(duration=0.5, freq=1000, sample_rate=44100):
    t = np.linspace(0, duration, int(sample_rate * duration), False)
    note = np.sin(freq * t * 2 * np.pi)
    audio = note * (2**15 - 1) / np.max(np.abs(note))
    audio = audio.astype(np.int16)
    buffer = io.BytesIO()
    sf.write(buffer, audio, sample_rate, format='WAV')
    buffer.seek(0)
    return (buffer.getvalue(), 'audio/wav')

def car_check_interface(voice_input, face_image, gps_location, enable_sms, fast2sms_api_key, to_phone):
    if voice_input is None and face_image is None:
        return "Welcome to your car's safety system. Say 'Ready to drive' (mic) or snap a face photo (cam) to start the full check.", None, None

    # Fixed: Mock transcription (simulate real speech; replace with Whisper for actual STT)
    mock_phrases = ["ready to drive", "i am sober", "help me", "feeling drowsy", "im fine", "madad karo"]
    transcribed = random.choice(mock_phrases) if voice_input else "face scan only"
    audio_path = voice_input if voice_input else "/tmp/voice_check.wav"

    # Audio analysis (drunk detection)
    transcribed_lower = transcribed.lower()
    if 'help' in transcribed_lower or 'madad' in transcribed_lower:
        audio_probs = {'normal': 0.06, 'slurred': 0.0, 'stressed': 0.0, 'help': 0.94}
        help_detected = True
        audio_unfit = 0.94
    elif len(transcribed.split()) < 2 or random.random() < 0.05:
        audio_probs = {'normal': 0.4, 'slurred': 0.4, 'stressed': 0.2, 'help': 0.0}
        help_detected = False
        audio_unfit = 0.6
    else:
        audio_probs = {'normal': 0.94, 'slurred': 0.0, 'stressed': 0.0, 'help': 0.06}
        help_detected = False
        audio_unfit = 0.06

    # Camera Analysis (Visual Impairment Detection)
    visual_unfit = 0.0  # 0-100% risk from face/eyes
    if face_image is not None:
        # Mock analysis (in real: Use OpenCV/MediaPipe to detect eye redness, blinks, head tilt)
        img_buffer = io.BytesIO()
        face_image.save(img_buffer, format='PNG')
        img_data = base64.b64encode(img_buffer.getvalue()).decode()
        if len(img_data) < 10000 or random.random() < 0.3:  # Mock "red eyes" or "unsteady gaze"
            visual_unfit = random.uniform(0.4, 0.8)  # 40-80% risk
            print("Debug: Camera detected potential visual impairment (e.g., red eyes or sway).")
        else:
            visual_unfit = 0.05  # Low risk
        face_note = f"Face scan: {visual_unfit:.0%} visual risk (eyes/gaze stable)."
    else:
        face_note = "No face scan – Using voice/sensors only."

    # Simulate motion
    motion_unfit = 0.01 if not help_detected else random.uniform(0.8, 0.9)

    # Fuse all: Audio + Visual + Motion (weighted: 40% voice, 40% cam, 20% motion)
    noise_level = 0.1
    for k in audio_probs:
        audio_probs[k] *= (1 - noise_level)
    audio_unfit = sum([audio_probs[k] for k in ['slurred', 'stressed', 'help']])
    drunk_score = (0.4 * audio_unfit + 0.4 * visual_unfit + 0.2 * motion_unfit)
    is_unfit = drunk_score > 0.7 or help_detected or visual_unfit > 0.6
    decision = "ENGINE STARTS – Safe to Drive" if not is_unfit else "IGNITION LOCKED – Do Not Drive"

    result = {
        'audio_probs': audio_probs, 'audio_unfit': audio_unfit, 'motion_unfit': motion_unfit, 'visual_unfit': visual_unfit,
        'drunk_score': drunk_score, 'is_unfit': is_unfit, 'help_detected': help_detected,
        'decision': decision
    }

    clean_probs = {k: round(v, 2) for k, v in audio_probs.items()}

    # Trigger SOS if unfit
    if is_unfit:
        trigger_sos(
            result=result, audio_path=audio_path, gps_location=gps_location,
            enable_sms=enable_sms, fast2sms_api_key=fast2sms_api_key,
            from_phone=None, to_phone=to_phone
        )

    # Generate beep sound on alert
    beep_audio = generate_beep() if is_unfit else None

    # Immersive output
    status = "🚗 Engine Ready – Drive Safely" if not is_unfit else "🔒 Ignition Locked – Assistance Needed"

    explanation = (
        f"Checks complete: Voice '{transcribed[:20]}...', {face_note}. All sensors stable – Proceed safely."
        if not is_unfit else
        f"Issue detected: Voice '{transcribed[:20]}...', {face_note}, unstable motion. Locked for safety."
    )
    if help_detected:
        explanation += " Emergency: 'Help' in voice – Alert sent."
    if visual_unfit > 0.5:
        explanation += " Visual: Eyes/gaze indicate impairment – Rest advised."

    edge_note = "Multi-sensor sim (voice + cam + motion). In real car: Dashboard cam auto-scans on startup. Privacy: Face data deleted post-check."

    output = f"""
**Car Status:** {status}

**Voice Analysis:** {audio_unfit:.0%} risk (from '{transcribed}')
**Breakdown:** Normal: {clean_probs['normal']:.0%} | Slurred: {clean_probs['slurred']:.0%} | Stressed: {clean_probs['stressed']:.0%} | Help: {clean_probs['help']:.0%}

**Camera Scan:** {visual_unfit:.0%} visual risk (eyes/face stable? {face_note})

**Sensor Motion Risk:** {motion_unfit:.0%} unstable

**Overall Safety Score:** {drunk_score:.0%} (Fused: Voice 40% + Cam 40% + Motion 20%; Under 70% = Go)

**Distress Detected?** {'Yes' if help_detected else 'No'}
**Ignition Decision:** {decision}
**Ready to Drive?** {'Yes – Buckle up!' if not is_unfit else 'No – Call for help'}

{explanation}

{edge_note}

**Alert Log:** Check Colab (GPS: {gps_location}). SMS if enabled.
**Note:** Local sim; real app uses car cam for eye tracking (e.g., pupil dilation). { '🚨 BEEP PLAYING – ALERT!' if is_unfit else '' }
"""
    return output, transcribed, beep_audio

# Enhanced Dashboard UI (With Camera + Alert Sound)
with gr.Blocks(theme=gr.themes.Soft(), title="Car Safety System with Camera") as demo:
    gr.Markdown("# 🚗 In-Car Safety Assistant (Voice + Face Cam + Sensors)")
    gr.Markdown("""
    **Driver Mode Activated!** Full check: Speak + Face scan for impairment (eyes, slurring, stability).
    **Your Flow:**
    1. Click mic & say 'Ready to drive' (or 'Help' for demo).
    2. Snap face photo (webcam – front view, eyes open).
    3. Hit 'Start Engine Check' – Auto-fuses all. Safe? Engine on. Risky? Locked + alert + BEEP!
    **Why Camera?** Detects red eyes/unsteady gaze (what voice misses). Real car: Dash cam auto-runs.
    **Test:** Safe = Clear speech + normal face photo (no beep). Risk = 'Help' + squint/tired pose (beep plays!).
    """)

    with gr.Row():
        with gr.Column(scale=2):  # Main Inputs
            voice_input = gr.Audio(label="🔊 Voice Check (Say 'Ready to drive' – 2-3s)",
                                   type="filepath", sources=["microphone"])
            face_input = gr.Image(label="📷 Face Scan (Click webcam – Front view, eyes visible; detects impairment)",
                                  sources=["webcam"], type="pil", height=200)
            ignition_btn = gr.Button("🔑 Start Engine & Full Check", variant="primary", size="lg")
            output_display = gr.Markdown(label="Dashboard")
            voice_echo = gr.Textbox(label="Transcribed Speech", interactive=False)
            alert_sound = gr.Audio(label="🚨 Alert Sound (Beep on Risk)", interactive=False, autoplay=True)  # New: Plays beep on alert

        with gr.Column(scale=1):  # Sidebar
            gps_input = gr.Textbox(label="GPS (Auto from Car/Phone)", value="19.0760, 72.8777")

    # Settings
    with gr.Accordion("⚙️ Alerts & Privacy", open=False):
        gr.Markdown("SMS to contacts on risk. Camera: Local only, no storage. Beep: Plays on >70% risk.")
        enable_sms = gr.Checkbox(label="Enable Alerts", value=False)
        fast2sms_api_key = gr.Textbox(label="API Key", type="password")
        to_phone = gr.Textbox(label="Emergency Number")

        gr.Markdown("""
        **Privacy Boost:** Face image processed locally (no upload). Delete via browser. GDPR-ready for cars.
        """)

    # Link Function (Now outputs sound too)
    ignition_btn.click(fn=car_check_interface,
                       inputs=[voice_input, face_input, gps_input, enable_sms, fast2sms_api_key, to_phone],
                       outputs=[output_display, voice_echo, alert_sound])

    gr.Markdown("""
    **Tips:**
    - Safe: Say 'ready' + clear face photo → 5-10% score, engine starts (no beep).
    - Risk: Say 'help' + tired/squint photo → 80%+, locked + BEEP + alert (check Colab log).
    - Real Upgrade: Integrate with car cam (e.g., via OBD + ML for 95% accuracy). Logs: `!cat /content/sos_log.txt`.
    - Why Cam Rocks: Catches visual impairment (e.g., no slurring but red eyes). Test now! Sound: Click if browser blocks autoplay.
    """)

# Launch
demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://73483915032006033d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://73483915032006033d.gradio.live


In [11]:
!pip install gradio==5.47.2